# Evidence-plus-answer SFT: Qwen2.5-0.5B

Continue training the existing strict-extractive **best supported-dev MRR** 0.5B adapter on the two exported annotation versions. Each run starts independently from the same initial adapter. Original source models and exports are preserved.

Defaults: both datasets, evidence-plus-answer targets, NF4 loading, existing LoRA rank 32/alpha 32/dropout 0.05, learning rate 5e-5, batch size 1, accumulation 32, up to six additional epochs. Stop after three checkpoint evaluations without improved all-dev MRR. This trainer uses Transformers/PEFT and standard gradient checkpointing, rather than Unsloth. These are new experiments with changed prompts/context budgets, not exact reproductions of the historical evaluator.

Run in the `bioasq` environment with a CUDA GPU. Preflight needs only CPU/tokenizer files and makes no judge/API requests. Training starts only in the Train cell.

**Dataset differences:** judge = 825 examples/questions, sufficient resource IDs, shuffled resources; occurrence = 1,231 examples / 1,062 questions, all alias-containing snippet IDs, original order. These full exports are not a controlled annotation-quality comparison. `ARMS=["evidence", "answer"]` adds a paired answer-only continuation control for each dataset.

Dev generation uses one shared evidence-plus-answer prompt for every model: all 160 original dev questions, individual snippets numbered in original resource order, including unsupported questions. Training examples retain their original variant-specific prompts and labels. At step 0 and every epoch checkpoint, the trainer records dev MRR, fixed-label evidence correctness, grounded accuracy, format validity, and teacher-forced loss on one deterministic annotated evidence-plus-answer target per eligible dev question. Unlisted snippets and sets remain unreviewed rather than negative.

Default mode is now **train**. It starts or resumes the two fresh common-evaluation runs. Set `RUN_MODE="evaluate_existing"` only to evaluate the initial SFT adapter and the previously selected adapters. Changing evaluation format does not remove the differences between the training datasets.


In [ ]:
from pathlib import Path
import sys, json, subprocess
from datetime import datetime, timezone

def find_root(start):
    for parent in [start, *start.parents]:
        for candidate in [parent, parent / "Task-Structured Counterfactual Preference Mining"]:
            if (candidate / "cse_dpo/train_factoid_evidence_answer_sft.py").exists():
                return candidate.resolve()
    raise RuntimeError("Open this notebook inside the project")
PROJECT_ROOT = find_root(Path.cwd())
print("Python:", sys.executable)
print("Project:", PROJECT_ROOT)


## Configuration

Keep the same `RUN_NAME` to resume interrupted runs automatically, including optimizer/scheduler state. Completed runs are skipped. Change it for a new experiment or changed hyperparameters. The default six epochs are additional training on these exports. Set `FROM_BASE=True` only to start a new LoRA adapter from Qwen2.5-0.5B-Instruct instead of continuing the trained SFT adapter.

`MAX_SEQUENCE_LENGTH=0` automatically chooses a sufficient token budget for the complete training examples and dev prompts plus generation. No examples are silently truncated or discarded. If memory is insufficient, do not lower the budget and silently crop evidence; change the experiment explicitly.

In [ ]:
VARIANTS_TO_TRAIN = ["judge", "occurrence"]
ARMS = ["evidence"]  # Add "answer" for matched answer-only controls within each dataset.
RUN_MODE = "train"  # "train" or "evaluate_existing"
SOURCE_RUN_NAME = "evidence_answer_sft_05b_v1"
RUN_NAME = "evidence_answer_sft_05b_common_eval_v5_official_java"  # New loss/evidence curves; do not mix with earlier histories.
FROM_BASE = False
INITIAL_ADAPTER = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr"
BASE_MODEL = PROJECT_ROOT / "models/Qwen2.5-0.5B-Instruct"
EXPORT_DIRS = {
    "judge": PROJECT_ROOT / "Artifacts/Factoid_SFT/evidence_annotation/supported_train_evidence_v1/exports/20260910_040647_418485",
    "occurrence": PROJECT_ROOT / "Artifacts/Factoid_SFT/evidence_occurrence/20260910_040958_563331",
}
DEV_EVIDENCE_EXPORT = PROJECT_ROOT / "Artifacts/Factoid_SFT/dev_evidence_annotation/full_dev_evidence_train_aligned_v2/exports/20260910_101557_748928"
EPOCHS = 6
LEARNING_RATE = 5e-5  # Conservative continuation rate; old SFT trained with 6e-4.
GRADIENT_ACCUMULATION = 32
MAX_SEQUENCE_LENGTH = 0
MAX_NEW_TOKENS = 512  # Accommodate occurrence targets listing many snippet IDs.
EARLY_STOPPING_PATIENCE = 3  # Set 0 to disable.
SEED = 3407
LOAD_IN_4BIT = True
OUTPUT_ROOT = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_answer_qwen25_05b" / RUN_NAME
assert RUN_NAME and RUN_NAME not in {".", ".."} and not any(c in RUN_NAME for c in "/\\")
assert VARIANTS_TO_TRAIN and set(VARIANTS_TO_TRAIN) <= set(EXPORT_DIRS)
assert ARMS and set(ARMS) <= {"evidence", "answer"}

def command_for(variant, arm, preflight=False):
    output = OUTPUT_ROOT / f"{variant}_{arm}_seed{SEED}"
    cmd = [sys.executable, "-u", "-m", "cse_dpo.train_factoid_evidence_answer_sft",
           "--variant", variant, "--arm", arm, "--export-dir", str(EXPORT_DIRS[variant]),
           "--base-model", str(BASE_MODEL), "--initial-adapter", str(INITIAL_ADAPTER),
           "--dev-evidence-export", str(DEV_EVIDENCE_EXPORT),
           "--output-dir", str(output), "--epochs", str(EPOCHS),
           "--learning-rate", str(LEARNING_RATE), "--grad-accum", str(GRADIENT_ACCUMULATION),
           "--max-seq-length", str(MAX_SEQUENCE_LENGTH), "--max-new-tokens", str(MAX_NEW_TOKENS),
           "--patience", str(EARLY_STOPPING_PATIENCE), "--seed", str(SEED)]
    if FROM_BASE: cmd.append("--from-base")
    if not LOAD_IN_4BIT: cmd.append("--no-4bit")
    if preflight: cmd.append("--preflight-only")
    return cmd, output
print("Experiments:", [(v,a) for v in VARIANTS_TO_TRAIN for a in ARMS])
print("Output:", OUTPUT_ROOT)

assert RUN_MODE in {"evaluate_existing", "train"}
SOURCE_ROOT = PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_answer_qwen25_05b" / SOURCE_RUN_NAME
EVAL_OUTPUT_ROOT = PROJECT_ROOT / "Artifacts/Factoid_SFT/evaluations" / (
    "evidence_answer_common_prompt_05b_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f"))
EVAL_TARGETS = {"initial_sft": ("judge", "evidence", INITIAL_ADAPTER)}
for variant in VARIANTS_TO_TRAIN:
    for arm in ARMS:
        EVAL_TARGETS[f"{variant}_{arm}_previous_best"] = (
            variant, arm, SOURCE_ROOT / f"{variant}_{arm}_seed{SEED}" / "adapter_best_mrr")
print("Run mode:", RUN_MODE)
print("Shared evaluation: Evidence: [snippet IDs] then Answer: [BE]answer[EE]")
print("Evaluation outputs:", EVAL_OUTPUT_ROOT)


## CPU preflight

Checks training and dev-annotation export hashes, unique training IDs, train/dev separation, target format, assistant-only loss masks, sufficient-set IDs, and exact token lengths. No model weights or API key are loaded. Review the counts and maximum lengths before training.

In [ ]:
for variant in VARIANTS_TO_TRAIN:
    for arm in ARMS:
        cmd, output = command_for(variant, arm, preflight=True)
        subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

reports = []
for variant in VARIANTS_TO_TRAIN:
    for arm in ARMS:
        _, output = command_for(variant, arm)
        reports.append(json.loads((output / "preflight.json").read_text()))
assert len({r["eval_inputs_sha256"] for r in reports}) == 1, "Evaluation prompts differ across variants"
assert len({r["dev_loss_targets_sha256"] for r in reports}) == 1, "Dev-loss targets differ across variants"
print("Verified identical evaluation prompt tokens across all configured variants/arms.")
if RUN_MODE == "evaluate_existing":
    for name, (_, _, adapter) in EVAL_TARGETS.items():
        assert (adapter / "adapter_model.safetensors").exists(), f"Missing adapter: {name}: {adapter}"


## Evaluate existing adapters or train

In `evaluate_existing` mode, the next cell evaluates the original SFT adapter once and both previously selected trained adapters. All use the **same new evidence-plus-answer system prompt, numbered snippets in original order, 512-token generation budget, greedy decoding, the official BioASQ Java scorer, frozen evidence labels, and frozen dev-loss targets**. The saved input hashes must match across models. A lone `[BE]answer[EE]` can still be scored but is invalid for the requested two-line format.

Evaluation-only results go to a new timestamped directory. The previous best adapters were selected under the old evaluation prompts; this comparison does not reselect the best epoch under the shared prompt. It does not update their weights.

Set `RUN_MODE="train"` for new training runs. These retain variant-specific training annotations and now select checkpoints using the common evaluation prompt. The new `RUN_NAME` keeps their metrics separate from older runs. Future resumes must keep the same evaluation protocol. Checkpoints are saved each epoch.

Shut down other kernels holding GPU models before launching. Processes run sequentially. A 10 GiB free-memory launch check is not a guarantee that every sequence fits. Training computes vocabulary scores only for supervised target positions while keeping the full context.


In [ ]:
import torch
import gc

MIN_FREE_GPU_GIB = 10.0  # Launch check for these full-context runs; not a peak-memory estimate.
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable in this kernel. Use the bioasq environment on your GPU machine.")
print("GPU:", torch.cuda.get_device_name(0))
def check_free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    free_bytes, total_bytes = torch.cuda.mem_get_info(0)
    print(f"GPU memory: {free_bytes / 2**30:.2f} / {total_bytes / 2**30:.2f} GiB free", flush=True)
    if free_bytes / 2**30 < MIN_FREE_GPU_GIB:
        try:
            subprocess.run(["nvidia-smi", "--query-compute-apps=pid,process_name,used_gpu_memory",
                            "--format=csv"], check=False)
        except FileNotFoundError:
            pass
        raise RuntimeError("Shut down unused GPU model kernels/processes before rerunning this cell.")

if RUN_MODE == "evaluate_existing":
    for name, (variant, arm, adapter) in EVAL_TARGETS.items():
        check_free_gpu()
        cmd, _ = command_for(variant, arm)
        cmd[cmd.index("--output-dir") + 1] = str(EVAL_OUTPUT_ROOT / name)
        cmd.extend(["--evaluate-only", "--eval-adapter", str(adapter)])
        print(f"Evaluating {name}: {adapter}", flush=True)
        subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)
else:
    for variant in VARIANTS_TO_TRAIN:
        for arm in ARMS:
            check_free_gpu()
            cmd, output = command_for(variant, arm)
            print(f"Starting {variant}/{arm}: {output}", flush=True)
            subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)


## Results

`adapter_best_mrr` is selected by all-dev answer MRR, `adapter_best_supported_mrr` by the 129-question occurrence-supported dev subset, `adapter_best_grounded` by exact answer plus a known sufficient evidence set, and `adapter_final` is the final training state. The baseline is eligible for selection, so a best adapter can remain the initial SFT model if training does not improve MRR. Check `selection.json` for step 0 versus trained checkpoints.

`generated_dev/` retains raw predictions, parsed answers, evidence IDs, format validity, answer scores and fixed-label evidence sufficiency. `evidence_correctness_score` is the fraction of exact-correct, annotation-covered answers whose cited IDs contain a known sufficient set. `grounded_dev_accuracy` uses all 160 questions as its denominator. Neither metric marks additional unreviewed citations wrong. `eval_loss` is the mean per-question teacher-forced completion loss on one frozen eligible evidence-plus-answer target per annotated question.

`metrics_history.json` stores every logged training loss and every checkpoint's eval loss, MRR, evidence correctness, grounded accuracy and format rate. `dev_loss_targets.jsonl`, `dev_inputs.jsonl`, `train_ids.jsonl`, `run_manifest.json`, and `preflight.json` preserve provenance. No API or LLM judge is called.

For `evaluate_existing`, each model directory contains `summary.json`, `predictions.jsonl`, `dev_inputs.jsonl`, and `run_manifest.json`. Compare these fresh summaries to one another, not to the old histories. Identical evaluation prompts do not make the training datasets matched; the judge arm was trained with resource IDs whereas this evaluation requests snippet IDs.


In [ ]:
if RUN_MODE == "evaluate_existing":
    summaries = []
    for name in EVAL_TARGETS:
        path = EVAL_OUTPUT_ROOT / name / "summary.json"
        if path.exists():
            result = json.loads(path.read_text())
            summaries.append(result)
            print(name, result)
        else:
            print(name, "No completed shared-prompt evaluation yet.")
    if summaries:
        assert len({r["eval_inputs_sha256"] for r in summaries}) == 1
    print("Evaluation directory:", EVAL_OUTPUT_ROOT)
else:
    for variant in VARIANTS_TO_TRAIN:
        for arm in ARMS:
            _, output = command_for(variant, arm)
            print(f"\n{variant}/{arm}")
            history = output / "generated_dev/history.json"
            if not history.exists():
                print("No completed dev evaluation yet.")
                continue
            for row in json.loads(history.read_text()):
                print(row)
            for name in ["adapter_best_mrr", "adapter_best_supported_mrr", "adapter_best_grounded"]:
                path = output / name / "selection.json"
                if path.exists(): print(name, json.loads(path.read_text()), "Path:", path.parent)


## Training curves

Run after training. Each experiment reads its persisted `metrics_history.json` and saves `training_curves.png` beside the adapters. Training loss is logged at every optimizer step; dev loss and generated metrics are measured at step 0 and each epoch checkpoint.

In [ ]:
if RUN_MODE == "train":
    import matplotlib.pyplot as plt
    for variant in VARIANTS_TO_TRAIN:
        for arm in ARMS:
            _, output = command_for(variant, arm)
            path = output / "metrics_history.json"
            if not path.exists():
                print(f"{variant}/{arm}: no metrics history yet.")
                continue
            metrics = json.loads(path.read_text())
            train = metrics["training"]
            dev = metrics["validation"]
            fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
            if train:
                axes[0, 0].plot([r["step"] for r in train], [r["train_loss"] for r in train])
            axes[0, 0].set(title="Training loss", xlabel="Optimizer step", ylabel="Loss")
            if dev:
                steps = [r["step"] for r in dev]
                axes[0, 1].plot(steps, [r["eval_loss"] for r in dev], marker="o")
                axes[1, 0].plot(steps, [r["dev_mrr"] for r in dev], marker="o", label="All-dev MRR")
                axes[1, 0].plot(steps, [r["supported_dev_mrr"] for r in dev], marker="o", label="Supported-dev MRR")
                axes[1, 1].plot(steps, [r["evidence_correctness_score"] for r in dev], marker="o", label="Evidence correctness")
                axes[1, 1].plot(steps, [r["grounded_dev_accuracy"] for r in dev], marker="o", label="Grounded dev accuracy")
            axes[0, 1].set(title="Annotated dev loss", xlabel="Optimizer step", ylabel="Loss")
            axes[1, 0].set(title="Answer quality", xlabel="Optimizer step", ylabel="Score", ylim=(0, 1))
            axes[1, 1].set(title="Evidence quality", xlabel="Optimizer step", ylabel="Score", ylim=(0, 1))
            for ax in axes.flat:
                ax.grid(alpha=0.25)
            axes[1, 0].legend()
            axes[1, 1].legend()
            fig.suptitle(f"{variant}/{arm}")
            figure_path = output / "training_curves.png"
            fig.savefig(figure_path, dpi=160)
            plt.show()
            print("Saved:", figure_path)
